<div>
<center><img src="../assets/Flux-logo.svg" width="400"/>
</div>

# Chapter 4: Flux with User-space Kubernetes

We are developing a new setup where it is possible to run Flux alongside user-space Kubernetes, or Usernetes.
This is a small demo of what that setup affords. In this tutorial, we will:
1. Start Usernetes
2. Run an AI/ML training Job 
3. Run the Flux Operator (Flux -> Kubernetes -> Flux)
4. Run a MuMMI State Machine
<br>

If you haven't already, open the [usernetes workspace](usernetes-workspace.jupyterlab-workspace) to have a working terminal alongside this notebook!

## 1. Start Usernetes

Normally, we start Usernetes under a Flux batch job, meaning you (the user) do not have to do it, and do not see it. Here, we are going to show you all the setup. Run the following script <strong>in the terminal to your left</strong> to watch your control plane come up!

```bash
bash /home/ubuntu/start-usernetes.sh
```

When the script finishes, you'll export the `KUBECONFIG`. This is your credentials to interact with the cluster.

```bash
export KUBECONFIG=/home/ubuntu/usernetes/kubeconfig
```

And try looking at the nodes. 

```bash
kubectl get nodes
```

Right now you just have a single control plane, and we have modified it to allow for running work. This is how we map Usernetes nodes to physical nodes in HPC, with a 1:1 ratio. Finally, you can enable auto-complete (with TAB) for `kubectl`:

```bash
source <(kubectl completion bash)
```

Take a look at all the service pods! In Kubernetes we have the concept of [namespaces](https://kubernetes.io/docs/concepts/overview/working-with-objects/namespaces/) to organize things. By default we interact with `default`.

```bash
kubectl get pods --all-namespaces 
```


## 2. Run an AI/ML Training Job

<div class="alert alert-block" style="background-color:skyblue">
<span style="font-weight:600">Description:</span> Running AI/ML training jobs in Kubernetes is a first class citizen.
</div>

The newly released [Kubeflow Trainer](https://www.kubeflow.org/docs/components/trainer/getting-started/) project makes it easy to run Kubernetes components, specifically for Artificial Intelligence and Machine Learning workloads (AI/ML) in Kubernetes directly from Python. In your terminal to the left, use `kubectl` to install the Kubeflow Training Operator: 

```bash
# This is for JobSet and the Trainer Manager
VERSION=v2.0.0
kubectl apply --server-side -k "https://github.com/kubeflow/trainer.git/manifests/overlays/manager?ref=${VERSION}"

# This is for the runtimes
kubectl apply --server-side -k "https://github.com/kubeflow/trainer.git/manifests/overlays/runtimes?ref=${VERSION}"
```

If you get a `failed calling webhook` error, wait 30 seconds and try again. Hooks sometimes have a delay in starting up.
Next, use the kubeflow trainer Python sdk to see the available training runtimes. Do you like Python and dislike YAML? While we won't use it in this tutorial, the entire interaction of using Kubeflow in Kubernetes can be done using the [Python SDK](https://www.kubeflow.org/docs/components/trainer/getting-started/).

In [2]:
# Check available training runtimes
from kubeflow.trainer import TrainerClient, CustomTrainer

import os
os.putenv("KUBECONFIG", "/home/ubuntu/usernetes/kubeconfig")
for r in TrainerClient().list_runtimes():
    print(f"Runtime: {r.name}")

Runtime: deepspeed-distributed
Runtime: mlx-distributed
Runtime: mpi-distributed
Runtime: torch-distributed
Runtime: torchtune-llama3.2-1b
Runtime: torchtune-llama3.2-3b


Let's first create the job in the "old school" way - by applying a YAML file. Take a look at [pytorch-mnist.yaml](pytorch-mnist.yaml) and then in your terminal, run the job in your cluster by using `kubectl apply` with `-f` for a file.

```bash
kubectl apply -f ./pytorch-mnist.yaml
```

To see the pods, you can use `kubectl get` on the Pod resource type. Note that we will have two. Index 0 is the master, and 1 is the worker.

```bash
kubectl get pods

# More information about the hosts in "output wide" mode
kubectl get pods -o wide
```

And then get the pod identifier and look at the output. The `-f` will keep the output streaming.

```bash
kubectl logs pytorch-simple-node-0-0-xxxx -f
```

When the logs appear to be done, check the pods to see they are `Completed`

```bash
kubectl get pods
```

When you are ready to clean up, just `kubectl delete` the same file.

```bash
kubectl delete -f pytorch-mnist.yaml
```

Congratulations - you just ran your first AI/ML Job in User-space Kubernetes!

## 3. Run the Flux Operator

<div class="alert alert-block" style="background-color:skyblue">
<span style="font-weight:600">Description:</span> Deploy an entire HPC cluster (1 node) in Kubernetes with the Flux Operator.
</div>

For this step, we are going to use the Flux Operator to deploy Flux Framework to Kubernetes. 
If we had more than one physical node, this would give us a powerful means to run HPC workloads in Kubernetes, with features that HPC cannot easily support such as elasticity and dynamism, declarative management, and modularity.
We will run a component of a well-known complex workflow, MuMMI. This is a machine learning running `mlrunner` that generates simulation data to kick off a sequence of simulation and analysis steps. First, install the Flux Operator (ARM variant):

```bash
kubectl apply -f https://raw.githubusercontent.com/flux-framework/flux-operator/refs/heads/main/examples/dist/flux-operator-arm.yaml
```

Next, create the MuMMI component job. The Flux Operator can provide several means to create a cluster, persistent or running as a job, and we will do the second. This job will start and complete, just like the AI/ML workload.

```bash
kubectl apply -f flux-mummi-mlrunner.yaml
```

This container has a large model in it, and will be in `Init:0/1` and then `PodIniitalizing` to pull the container before it is `Running`. This takes
And again look at the logs for the created pod.

```bash
kubectl logs 
```